# ViT-LENS: Final Faculty Demonstration & Interactive Application Launch

**Project**: *Discovering and Causally Testing Interpretable Features in Vision Transformers Using TopK Sparse Autoencoders*

---

### Instructions for Quick Restart After Runtime Reset
This lightweight notebook is optimized for **Google Colab (CUDA GPU recommended)**.
It loads the already trained TopK SAE checkpoint and frozen pretrained ViT model without repeating long training or data extraction runs:
- ❌ **NO** SAE retraining
- ❌ **NO** Imagenette / ImageNet dataset downloading
- ❌ **NO** activation chunk regeneration
- ❌ **NO** experimental re-running
- ✅ **YES**: Google Drive checkpoint loading, architecture validation, and Gradio launch with temporary public share link (`share=True`).

### Step 1: Mount Google Drive & Set Working Directory

In [ ]:
import os
import sys

# Mount Google Drive to access persistent checkpoints and experiment results
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Mounted Google Drive successfully.")
except ImportError:
    print("Not running in Google Colab; skipping Drive mount.")

# Ensure repository root is on Python search path
repo_candidates = [
    "/content/ViT-LENS",
    "/content/drive/MyDrive/ViT-LENS",
    os.path.abspath(os.path.join(os.getcwd(), "..")),
    os.getcwd()
]
repo_root = None
for path in repo_candidates:
    if os.path.exists(os.path.join(path, "src", "topk_sae.py")):
        repo_root = path
        break

if repo_root:
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    os.chdir(repo_root)
    print(f"Set working directory to repository root: {repo_root}")
else:
    print("Current working directory:", os.getcwd())

### Step 2: Hardware Acceleration Check (GPU / CUDA)

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"CUDA is available: YES")
    print(f"Active GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    device = torch.device("cpu")
    print("CUDA is NOT available. Running on CPU.")
    print("Note: For optimal responsiveness during live demonstration, switch to a Colab GPU runtime (Runtime > Change runtime type > T4 GPU).")

### Step 3: Install Required Application Dependencies

In [ ]:
!pip install -q timm gradio matplotlib pillow pyyaml

### Step 4: Validate Checkpoint & Empirical Benchmarks

In [ ]:
from src.utils import resolve_sae_checkpoint_path
from src.evidence_service import EvidenceService

# Checkpoint resolution
ckpt_path = resolve_sae_checkpoint_path()
print(f"Target SAE Checkpoint Path: {ckpt_path}")
print(f"Checkpoint Exists: {os.path.exists(ckpt_path)}")

evidence_service = EvidenceService()
summary = evidence_service.get_summary()
benchmarks = summary["benchmarks"]
print("\n--- Established Empirical Results ---")
print(f"Explained Variance: {benchmarks['explained_variance']:.2%}")
print(f"Prediction Agreement: {benchmarks['reconstruction_agreement']:.2%}")
print(f"Feature #588 Class AUC (Tench/Fish): {benchmarks['class_auc']['feature_588']:.4f}")
print(f"Feature #1453 Class AUC (English Springer): {benchmarks['class_auc']['feature_1453']:.4f}")
print(f"Feature #2077 Class AUC (Fish): {benchmarks['class_auc']['feature_2077']:.4f}")

### Step 5: Initialize ModelService Session & Load Models

In [ ]:
from src.model_service import get_model_service

print("Initializing ModelService (loading frozen ViT-B/16 and TopK SAE)...")
service = get_model_service()
print(f"ViT Model: {service.vit_model_name}")
print(f"Target Residual Layer: Block {service.target_layer_idx}")
print(f"Compute Device: {service.device}")
print(f"SAE Loaded: {service.is_sae_loaded}")
if not service.is_sae_loaded:
    print(f"SAE Status Note: {service.sae_load_error}")

### Step 6: Launch Interactive Gradio Demonstration Web App
Run this cell to start the web application. In Google Colab, `share=True` generates a temporary public `.gradio.live` link so you can interact with the app on any browser or mobile device during your university presentation.

In [ ]:
from app.gradio_app import build_app

# Build and launch the Gradio user interface
demo = build_app(service=service)

# Launch with public share link for live university demonstration
demo.launch(
    share=True,
    inbrowser=True,
    show_error=True
)